# 1、前置说明
 - 标题结构化，层级顺序。

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
root_dir = r"E:\01_UCL\08_Dissertation\01_DataCollection\01 Place Pluse2.0数据集\00 place-pulse-dataset-images 原始数据集"
dest_dir = os.path.join(root_dir,"01代码运行产生的文件")
if not os.path.exists(dest_dir):
    os.makedirs(dest_dir, exist_ok=True)

# 1. locations.tsv
location_path = os.path.join(root_dir, 'locations.tsv')
locations_df = pd.read_csv(location_path, sep='\t')
print("locations.tsv:",locations_df.columns)
locations_df

In [ ]:
# locations_df['place_id'].unique()
locations_df['_id'].unique()

In [ ]:
locations_df = locations_df[['_id','place_id']]
locations_df

In [ ]:
temp_df = locations_df
temp_df

In [ ]:
# 2. places.tsv
places_path = os.path.join(root_dir, 'places.tsv')
places_df = pd.read_csv(places_path, sep='\t')
print("places.tsv:",places_df.columns)
places_df.head(100)

In [ ]:
places_df['_id'].unique()   #这里的_id就是地名的id，真无语，用_id,为何不干脆直接使用place_id

In [ ]:
places_df = places_df[['_id','place_name']]
places_df

In [ ]:
temp_df = temp_df.merge(places_df,left_on='place_id',right_on='_id')
temp_df

### 总结：命名规律，这个文件的_id，对应其他文件中列"文件名_id"

# 2.核心部分（qscores.tsv、studies.tsv、votes.tsv）

In [ ]:
# 3. qscores.tsv
qscores_path = os.path.join(root_dir, 'qscores.tsv')
qscores_df = pd.read_csv(qscores_path, sep='\t')
print("qscores.tsv:",qscores_df.columns)
qscores_df

In [ ]:
# qscores_df['_id'].unique()
# qscores_df['location_id'].unique()
# qscores_df['study_id'].unique()
# qscores_df['place_id'].unique()

In [ ]:
# qscores_df = qscores_df.drop(columns=['location_id','place_id'])
qscores_df = qscores_df[['location_id', 'num_votes', 'study_id','trueskill.score', 'trueskill.stds.-1']]
qscores_df

In [ ]:
merge_df = qscores_df
merge_df

In [ ]:
# 4. studies.tsv
studies_path = os.path.join(root_dir, 'studies.tsv')
studies_df = pd.read_csv(studies_path, sep='\t')
print("studies.tsv:",studies_df.columns)
studies_df

In [ ]:
studies_df = studies_df[['_id','study_question']]
studies_df

In [ ]:
merge_df = merge_df.merge(studies_df,left_on='study_id',right_on='_id')
print(merge_df.columns)
merge_df

In [ ]:
# merge_df = merge_df.drop(columns=['_id_y', 'study_id'])
merge_df = merge_df[['location_id', 'num_votes', 'trueskill.score','trueskill.stds.-1', 'study_question']]
merge_df

### trueskill.score归一化
### 这里的归一化也要看清你是针对某个维度的数据还是针对于所有维度的数据进行归一化，这里我是针对每个维度的数据。
1、如果不进行任何筛选，那就是每个维度所有的数据进行归一化。
2、但是当筛选>=12的数据时，那就是每个维度>=12的数据进行归一化。
3、两个归一化中，最大最小值可能是一样的，也有小概率不一样!！！当最大最小值不一样时，结果也不一样，所以用哪些数据进行归一化自己要想清楚。

In [ ]:
def normalize_group(df, target_min=0, target_max=10):
    min_val = df['trueskill.score'].min()
    max_val = df['trueskill.score'].max()
    study_question = df['study_question'].unique()  # 取当前分组的名称
    print(f"维度：{study_question}，最小值：{min_val}，最大值：{max_val}")
    if min_val == max_val:
        df['trueskill_norm'] = target_min  # 避免除以0，全部设为下限
    else:
        df['trueskill_norm'] = (
            (df['trueskill.score'] - min_val) / (max_val - min_val)
        ) * (target_max - target_min) + target_min
    return df

# 应用归一化
merge_df = merge_df.groupby('study_question', group_keys=False).apply(normalize_group)

In [ ]:
merge_df['trueskill_norm'] = merge_df['trueskill_norm'].round(2)
merge_df

In [ ]:
merge_df.groupby(by='study_question').size()

In [ ]:
merge_df.loc[merge_df['location_id'] == '5185d46bfdc9f03fd50013d5']

In [ ]:
merge_df.to_csv(os.path.join(dest_dir,"01CSV合并简化后的trueskill评分.csv"), index = False)

# 3.只筛选大于等于某个次数的数据

In [ ]:
merge_df

In [ ]:
min_votes = 12  # 设置投票数的最小阈值

merge_df_min_votes = merge_df[merge_df['num_votes'] >= min_votes]
merge_df_min_votes

In [ ]:
def normalize_group(df, target_min=0, target_max=10):
    min_val = df['trueskill.score'].min()
    max_val = df['trueskill.score'].max()
    study_question = df['study_question'].unique()  # 取当前分组的名称
    print(f"维度：{study_question}，最小值：{min_val}，最大值：{max_val}")
    if min_val == max_val:
        df['trueskill_norm'] = target_min  # 避免除以0，全部设为下限
    else:
        df['trueskill_norm'] = (
            (df['trueskill.score'] - min_val) / (max_val - min_val)
        ) * (target_max - target_min) + target_min
    return df

# 应用归一化
merge_df_min_votes = merge_df_min_votes.groupby('study_question', group_keys=False).apply(normalize_group)

In [ ]:
merge_df_min_votes.groupby(by='study_question').size()

In [ ]:
import matplotlib.pyplot as plt
# 遍历每个 study_question 的分组并保存
for question, group_df in merge_df_min_votes.groupby('study_question'):
    # ==== 保存 CSV ====
    filename = f"02对比次数大于等于{min_votes}的{question}维度数据.csv"
    file_path = os.path.join(dest_dir, filename)
    group_df.to_csv(file_path, index=False, encoding='utf-8-sig')
    print(f"已保存: {file_path}")

    # ==== 输出描述性统计信息 ====
    print(f"\n[{question}] 的 trueskill.score 描述性统计：")
    print(group_df['trueskill.score'].describe())

    # ==== 绘制直方图 ====
    plt.figure(figsize=(8, 5))
    plt.hist(group_df['trueskill.score'].dropna(), bins=30, color='skyblue', edgecolor='black')
    plt.title(f"Histogram of trueskill.score - {question} - Count: {len(group_df)}")
    plt.xlabel('trueskill.score')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.tight_layout()

    # 保存直方图
    hist_path = os.path.join(dest_dir, f"03对比次数大于等于{min_votes}的{question}_hist.png")
    plt.savefig(hist_path,dpi = 600)
    # plt.close()
    print(f"直方图已保存: {hist_path}")

# 4.筛选每个采样点，每个维度都大于等于某个数值的所有街景图片，没有数据满足，所以用一个模型是不合理的。

In [ ]:
# 第一步：统计每个 location_id 的出现次数
location_counts = merge_df_min_votes['location_id'].value_counts()
location_counts

In [ ]:
# 第二步：筛选出出现次数为 6 的 location_id
location_ids_6 = location_counts[location_counts == 6].index
location_ids_6

In [ ]:
# 第三步：从原始数据中筛选这些 location_id 对应的所有行
filtered_df = merge_df_min_votes[merge_df_min_votes['location_id'].isin(location_ids_6)]
filtered_df

### 归一化代码略

In [ ]:
filtered_df.to_csv(os.path.join(dest_dir,f"04每个街景图片6个维度的对比次数都大于等于{min_votes}的数据.csv"),index=False)

In [ ]:
# 5. votes.tsv
votes_path = os.path.join(root_dir, 'votes.tsv')
votes_df = pd.read_csv(votes_path, sep='\t')
print("votes.tsv:",votes_df.columns)
votes_df

In [ ]:
votes_df = votes_df[['_id','choice','left','right','study_id']]
votes_df